<a href="https://colab.research.google.com/github/hd07042014-maker/Movement-tracker/blob/main/Cash_Pop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q pypdf

from collections import Counter
import re
import pandas as pd
from pypdf import PdfReader
from google.colab import files
import os

def select_pdf_file():
    """Prompts the user to upload a PDF file using Colab's native file upload tool."""
    print("Please upload your Cash Pop PDF File:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".pdf"):
            return filename
    return None

def extract_cash_pop_data(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    draws = []
    pattern = re.compile(
        r"DRAW\s*#?\s*-?\s*(\d+)\s+(\d{2}/\d{2}/\d{4})\s+(\d+)\s+(\d+)\s+\$?([0-9,]+\.?\d*)",
        re.IGNORECASE
    )
    matches = pattern.findall(full_text)

    for match in matches:
        draw_id = int(match[0])
        date = match[1]
        winning_num = int(match[2])
        winners = int(match[3])
        amount_won = float(match[4].replace(",", ""))

        draws.append({
            "draw": draw_id,
            "date": date,
            "number": winning_num,
            "winners": winners,
            "amount_won": amount_won
        })

    return draws

def get_frequency_table(data, n):
    subset = data[:n]
    numbers = [item["number"] for item in subset]
    counts = Counter(numbers)

    full_counts = {i: counts.get(i, 0) for i in range(1, 16)}

    df = pd.DataFrame(
        list(full_counts.items()), columns=["Winning Number", "Frequency"]
    )
    df = df.sort_values(
        by=["Frequency", "Winning Number"], ascending=[False, True]
    ).reset_index(drop=True)
    return df

def get_stats_by_winning_number(data, n):
    """Calculates total/average winners and amounts won grouped by each Winning Number (1-15)"""
    subset = data[:n]

    # Initialize statistics for all numbers 1 to 15
    stats = {i: {"Total Winners": 0, "Total Amount Won": 0.0, "Draw Count": 0} for i in range(1, 16)}

    for item in subset:
        num = item["number"]
        if num in stats:
            stats[num]["Total Winners"] += item["winners"]
            stats[num]["Total Amount Won"] += item["amount_won"]
            stats[num]["Draw Count"] += 1

    # Build DataFrames
    winners_list = []
    amount_list = []

    for num, info in stats.items():
        winners_list.append({
            "Winning Number": num,
            "Total Winners": info["Total Winners"],
        })

        amount_list.append({
            "Winning Number": num,
            "Total Amount Won ($)": info["Total Amount Won"],
        })

    df_winners = pd.DataFrame(winners_list).sort_values(by="Total Winners", ascending=False).reset_index(drop=True)
    df_amount = pd.DataFrame(amount_list).sort_values(by="Total Amount Won ($)", ascending=False).reset_index(drop=True)

    # Format currency for output
    df_amount["Total Amount Won ($)"] = df_amount["Total Amount Won ($)"].apply(lambda x: f"${x:,.2f}")

    return df_winners, df_amount

def main():
    pdf_path = select_pdf_file()

    if not pdf_path:
        print("No PDF file uploaded. Program terminated.")
        return

    print(f"\nProcessing file: {pdf_path}\n")
    draws_data = extract_cash_pop_data(pdf_path)

    if os.path.exists(pdf_path):
        os.remove(pdf_path)

    if not draws_data:
        print("No valid draw data found in the selected PDF. Please check the regex pattern against your PDF layout.")
        return

    draws_data = sorted(draws_data, key=lambda x: x["draw"], reverse=True)

    df_50 = get_frequency_table(draws_data, 50)
    df_100 = get_frequency_table(draws_data, 100)
    df_150 = get_frequency_table(draws_data, 150)

    print("--- Latest 50 Draws Frequency Table ---")
    print(df_50.to_string(index=False))

    print("\n--- Latest 100 Draws Frequency Table ---")
    print(df_100.to_string(index=False))

    print("\n--- Latest 150 Draws Frequency Table ---")
    print(df_150.to_string(index=False))

    # Generate tables grouped by winning numbers for the last 50 draws
    df_winners_by_num, df_amount_by_num = get_stats_by_winning_number(draws_data, 30)

    print("\n--- Winners Distribution by Winning Number (Last 30 Draws) ---")
    print(df_winners_by_num.to_string(index=False))

    print("\n--- Money Won Distribution by Winning Number (Last 30 Draws) ---")
    print(df_amount_by_num.to_string(index=False))

if __name__ == "__main__":
    main()

Please upload your Cash Pop PDF File:


Saving 8b3b8661-8cd7-45b6-8f79-d50237c1f3ba.pdf to 8b3b8661-8cd7-45b6-8f79-d50237c1f3ba.pdf

Processing file: 8b3b8661-8cd7-45b6-8f79-d50237c1f3ba.pdf

--- Latest 50 Draws Frequency Table ---
 Winning Number  Frequency
              2          6
              7          5
              1          4
             11          4
             13          4
             15          4
              3          3
              4          3
              5          3
              6          3
              8          3
              9          3
             10          2
             12          2
             14          1

--- Latest 100 Draws Frequency Table ---
 Winning Number  Frequency
             12         10
              2          8
              5          8
              1          7
              3          7
              4          7
              7          7
              8          7
             10          7
             11          7
             13          7
          